# Reranking Architectures

**Module:** 03 — Reranking

Bi-encoders, cross-encoders, and hybrid rankers trade off scale, latency, and precision. Learn when each architecture belongs in the stack.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Explain bi-encoder dual-tower scoring and its limits
- Explain cross-encoder joint encoding for reranking
- Design hybrid pipelines (lexical + dense + CE)
- Reason about latency patterns and batching
- Pick an architecture given QPS, N, and quality targets


## Bi-Encoder

**Definition.** A **bi-encoder** embeds the query and document in separate towers; relevance is a similarity (dot / cosine) between vectors. Great for retrieval; weaker as a final reranker.

**Why it matters.** Embeddings can be precomputed for all docs—enabling ANN over millions of items.

**How it works.** Encode docs offline → index → encode query online → ANN top-N.

**Intuition.** Two people write summaries separately and you compare the summaries—not a live debate.

**Common pitfalls.**
- Expecting bi-encoder precision to match a cross-encoder
- Mixing embedding model versions in one index
- Forgetting to normalize when using cosine/IP conventions

**When to use.** First-stage retrieval and cheap candidate generation; sometimes light rescoring.

```mermaid
flowchart LR
  Q[Query] --> QE[Query encoder]
  D[Doc] --> DE[Doc encoder]
  QE --> IP[Dot / cosine]
  DE --> IP
  IP --> S[Score]
```


In [ ]:
# Demo 1 — dual tower scores
import numpy as np

def encode(text: str, dim=8, seed=0):
    rng = np.random.default_rng(abs(hash(text)) % (2**32) + seed)
    v = rng.normal(size=dim)
    return v / (np.linalg.norm(v) + 1e-9)

q = encode("refund window")
docs = ["60 day refunds", "shipping ETA", "password reset"]
for d in docs:
    print(d, round(float(encode(d) @ q), 3))


In [ ]:
# Demo 2 — bi-encoder training signal (contrastive sketch)
print("loss ~ -log softmax(sim(q, d+)/tau) among in-batch negatives")
print("needs hard negatives for strong rerank-quality bi-encoders")


In [ ]:
# Demo 3 — precompute vs online
n_docs, dim, qps = 1_000_000, 768, 50
print("offline bytes", n_docs * dim * 4 / 1e9, "GB float32")
print("online: 1 query encode + ANN; not N cross-encoder passes")


### Try it yourself — Bi-Encoder

1. Why can bi-encoders miss negation ('not eligible for refund')?
2. Name one case for a second bi-encoder rescoring pass.


## Cross-Encoder

**Definition.** A **cross-encoder** feeds the concatenated query and document into one Transformer and outputs a relevance score—full token-level attention across the pair.

**Why it matters.** Highest classical neural precision for shortlists; the default strong reranker class.

**How it works.** For each candidate: tokenize [q][SEP][d] → forward → logit/prob → sort. Batch pairs on GPU.

**Intuition.** The query and document actually talk to each other in the same room.

**Common pitfalls.**
- Running CE on the full corpus (latency explosion)
- Unbounded document length without truncation strategy
- Calibrating CE scores as probabilities across domains blindly

**When to use.** Second-stage rerank of N≈20–200 candidates when precision matters.

### Latency architecture patterns

| Pattern | Idea | Tradeoff |
|---------|------|----------|
| Sync CE | Rerank in request path | Simplest; watch p95 |
| Async CE | Prefetch/parallelize | Complexity |
| Cascades | Cheap filter → CE | Best cost/quality |
| Cache | Cache (q hash, doc id) scores | Great for head queries |


In [ ]:
# Demo 1 — pair formatting
def pair(q, d, max_chars=200):
    return f"Query: {q}\nDocument: {d[:max_chars]}"

print(pair("where did the cat sit?", "The cat sat on the mat in the living room."))


In [ ]:
# Demo 2 — batching pairs
candidates = [f"doc-{i}" for i in range(50)]
batch_size = 16
batches = [candidates[i:i+batch_size] for i in range(0, len(candidates), batch_size)]
print("num batches", len(batches), "last size", len(batches[-1]))


In [ ]:
# Demo 3 — truncation policy
def truncate(doc: str, max_tokens_est: int = 256):
    words = doc.split()
    return " ".join(words[:max_tokens_est]), max(0, len(words) - max_tokens_est)

text, dropped = truncate("word " * 400, 256)
print("kept_words", len(text.split()), "dropped", dropped)


In [ ]:
# Demo 4 — score then sort API shape
import json
YOUR_API_KEY = "YOUR_API_KEY"
resp = {
  "model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
  "scores": [0.12, 0.88, 0.44],
}
order = sorted(range(3), key=lambda i: resp["scores"][i], reverse=True)
print(json.dumps({"scores": resp["scores"], "order": order}, indent=2))
print("key", YOUR_API_KEY[:8] + "...")


### Try it yourself — Cross-Encoder

1. Estimate pair forwards for N=80 at batch 16.
2. Propose a truncation strategy for long policies vs short FAQs.


## Hybrid Ranking

**Definition.** **Hybrid ranking** fuses multiple signals—lexical (BM25), dense (bi-encoder), metadata boosts, and a cross-encoder—often via RRF or learned fusion, then CE.

**Why it matters.** Real corpora mix exact IDs and paraphrases; one channel rarely dominates all query classes.

**How it works.** Retrieve from each channel → fuse to N → optional CE → pack. Tune fusion weights on a labeled set.

**Intuition.** A relay team: sprinters fetch, strategist reorders.

**Common pitfalls.**
- Fusion without normalization (BM25 vs cosine scales)
- CE that undoes good lexical matches on SKU queries
- Too many stages → un-debuggable systems

**When to use.** Default for production RAG over heterogeneous knowledge bases.

```mermaid
flowchart TB
  Q[Query] --> BM25
  Q --> Dense
  BM25 --> RRF[RRF / fusion]
  Dense --> RRF
  RRF --> CE[Cross-encoder]
  CE --> TopK[Top-k]
```


In [ ]:
# Demo 1 — Reciprocal Rank Fusion
def rrf(rank_lists, k=60):
    scores = {}
    for lst in rank_lists:
        for rank, doc in enumerate(lst, 1):
            scores[doc] = scores.get(doc, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=scores.get, reverse=True)

bm25 = ["d1", "d2", "d9", "d3"]
dense = ["d3", "d1", "d4", "d2"]
print(rrf([bm25, dense])[:4])


In [ ]:
# Demo 2 — cascade
def cascade(lexical_top, dense_top, ce_score):
    pool = list(dict.fromkeys(lexical_top + dense_top))
    return sorted(pool, key=lambda d: ce_score(d), reverse=True)

ce = {"d1": 0.9, "d2": 0.2, "d3": 0.8, "d4": 0.4}.get
print(cascade(["d1", "d2"], ["d3", "d1"], ce))


In [ ]:
# Demo 3 — weighted fusion after score norm
def minmax(xs):
    lo, hi = min(xs), max(xs)
    return [(x - lo) / (hi - lo + 1e-9) for x in xs]

bm25_s = [12.0, 8.0, 3.0]
dense_s = [0.81, 0.77, 0.22]
fused = [0.4*b + 0.6*d for b, d in zip(minmax(bm25_s), minmax(dense_s))]
print([round(x, 3) for x in fused])


In [ ]:
# Demo 4 — architecture decision table as data
rows = [
    ("bi only", "low latency", "weak precision"),
    ("CE only on N", "best shortlist", "needs good recall@N"),
    ("hybrid+CE", "robust", "more moving parts"),
]
for a, b, c in rows:
    print(f"{a:14s} | {b:14s} | {c}")


### Try it yourself — Hybrid Ranking

1. Design a hybrid for SKU + natural language support search.
2. When would you skip CE and stop at RRF?


## Glossary

- **bi-encoder**: Dual-tower embedding similarity model
- **cross-encoder**: Joint query-doc Transformer scorer
- **RRF**: Reciprocal Rank Fusion across rank lists
- **cascade**: Cheap filters before expensive scorers


### Workshop drill — Reranking Architectures (1)

Restate each major section heading as one exam-ready sentence.


In [ ]:
# Workshop drill 1 — Reranking Architectures
headings = ['Bi-Encoder', 'Cross-Encoder', 'Hybrid Ranking']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Reranking Architectures (2)

Sketch a latency budget: first-stage ms + rerank (N candidates × cost) + LLM.


In [ ]:
# Workshop drill 2 — Reranking Architectures
first_ms, per_pair_ms, n, llm_ms = 40, 3, 50, 800
print('total_ms', first_ms + n*per_pair_ms + llm_ms)
print('rerank_share', round(n*per_pair_ms/(first_ms+n*per_pair_ms+llm_ms), 3))


### Workshop drill — Reranking Architectures (3)

Design an offline metric slice: 5 queries with graded relevance labels.


In [ ]:
# Workshop drill 3 — Reranking Architectures
eval_set = [{'q':'...','docs':{'d1':2,'d2':1,'d3':0}}]
print('n_queries', len(eval_set))
print('TODO: fill real labels')


### Workshop drill — Reranking Architectures (4)

Write a go/no-go checklist for shipping a reranker in RAG.


In [ ]:
# Workshop drill 4 — Reranking Architectures
for c in ['latency_p95','nDCG@10','cost/1k','cache_hit','fallback']:
    print(f'[ ] {c}')


## Summary & Key Takeaways

- Bi-encoders scale; cross-encoders precision-rerank shortlists
- Hybrid fusion covers lexical + semantic query classes
- Cascades and caches keep CE affordable
- Architecture choice is a latency/quality/cost triangle

### Practice

Implement RRF on two fake lists and feed the fused top-20 into a toy CE.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
